把set1-2分成train（fold0-4）和internal test（fold 5）。在train内部再用random_State形成不同的5-fold

In [1]:
import os
import numpy as np
import pandas as pd

from scipy import stats
from sklearn.model_selection import train_test_split, StratifiedKFold


# ============================================================
# Settings
# ============================================================

patient_list_path = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx"
out_root = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists"

# Clinical processed table contains the four tumor-size variables used to balance
# train/internal-test and evaluate CV-fold consistency.
clinical_variables_processed_path = os.path.join(
    out_root,
    "image_label_info_set12_clinical_variables_processed.xlsx",
)

task = "prognosis"
label_col = "Prognosis_label"

split_col = "split"
fold_col = "fold"

# Fixed train/internal-test split seed search starts here. The final selected
# seed may differ from this value if the tumor-size p-value pattern is not balanced.
split_random_state = 100
max_split_search_attempts = 20000

# We will search for 5 fold random seeds. A perfect seed has zero mismatch between
# each validation fold and its corresponding 4-fold CV-training subset in the
# tumor-size significant/non-significant pattern. If perfect seeds are rare, we keep
# the lowest-mismatch seeds.
num_fold_random_states_to_keep = 5
fold_seed_search_start = 0
max_fold_seed_search_attempts = 20000

n_splits = 5

# Required sizes: 232 train, 98 internal test from 330 total.
n_train = 232
n_internal_test = 98

# Tumor variables used for balancing/checking. These names come from
# image_label_info_set12_clinical_variables_processed.xlsx.
tumor_variable_cols = [
    "Tumor_AP_diameter_mm",
    "Tumor_longitudinal_diameter_mm",
    "Tumor_transverse_diameter_mm",
    "Tumor_volume_mm3",
]

tumor_p_alpha = 0.05

# Output audit table for the selected split/fold seeds.
tumor_balance_report_path = os.path.join(
    out_root,
    f"image_label_info_set12_5fold_{task}_tumor_balance_report.xlsx",
)


In [2]:
# ============================================================
# Load patient table and tumor-size clinical table
# ============================================================

df0 = pd.read_excel(patient_list_path)

print("Loaded:", patient_list_path)
print("Shape:", df0.shape)
print("Columns:", list(df0.columns))

if label_col not in df0.columns:
    raise ValueError(f"Missing label column: {label_col}")

if len(df0) != n_train + n_internal_test:
    raise ValueError(
        f"Expected {n_train + n_internal_test} cases, but found {len(df0)} cases."
    )

if df0[label_col].isna().any():
    raise ValueError(f"{label_col} contains missing values.")

df0[label_col] = df0[label_col].astype(int)
df0["Patient_set"] = df0["Patient_set"].astype(str)
df0["Patient_index"] = df0["Patient_index"].astype(str)

clinical_tumor_df = pd.read_excel(clinical_variables_processed_path)
clinical_tumor_df["Patient_set"] = clinical_tumor_df["Patient_set"].astype(str)
clinical_tumor_df["Patient_index"] = clinical_tumor_df["Patient_index"].astype(str)

required_clinical_cols = ["Patient_set", "Patient_index"] + tumor_variable_cols
missing_clinical_cols = [col for col in required_clinical_cols if col not in clinical_tumor_df.columns]
if missing_clinical_cols:
    raise KeyError(f"Missing columns in clinical processed table: {missing_clinical_cols}")

# Keep tumor features in a separate aligned dataframe. The saved split files will
# still be based on df0 and will not need extra tumor columns unless you decide to add them.
tumor_aligned_df = df0[["Patient_set", "Patient_index"]].merge(
    clinical_tumor_df[required_clinical_cols],
    on=["Patient_set", "Patient_index"],
    how="left",
    validate="one_to_one",
)

if tumor_aligned_df[tumor_variable_cols].isna().any().any():
    missing_cases = tumor_aligned_df.loc[
        tumor_aligned_df[tumor_variable_cols].isna().any(axis=1),
        ["Patient_set", "Patient_index"] + tumor_variable_cols,
    ]
    display(missing_cases)
    raise RuntimeError("Some cases are missing tumor-size variables.")

tumor_values = tumor_aligned_df[tumor_variable_cols].astype(float)

print("\nOverall label distribution:")
display(
    df0[label_col]
    .value_counts(dropna=False)
    .rename_axis(label_col)
    .reset_index(name="count")
)

overall_pos_frac = df0[label_col].mean()
print(f"Overall {label_col}=1 fraction: {overall_pos_frac:.4f}")

print("\nTumor variable table aligned to patient list:")
display(tumor_aligned_df.head())


Loaded: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12.xlsx
Shape: (330, 39)
Columns: ['Patient_set', 'Patient_index', 'Include', 'Have_seg', 'X_shape', 'Y_shape', 'Slice_num', 'Spacing', 'Image_filepath', 'Mask_filepath', 'Medical_record_number', 'Registration_number', 'Prognosis_label', 'Pathologic_label', 'Sort_order', 'Name', 'Pathologic_fracture', 'Length', 'Width', 'Height', 'Sex', 'Side', 'Lesion_site', 'Age', 'Height_at_visit (cm)', 'Weight_at_visit (kg)', 'WBC (*10^9/L)', 'HGB (g/L)', 'PLT (*10^9/L)', 'CRP (mg/L)', 'ALP (IU/L)', 'Total_cholesterol (mmol/L)', 'Triglycerides (mmol/L)', 'LDL (mmol/L)', 'LDH (IU/L)', 'PT (S)', 'APTT (S)', 'Fibrinogen (mg/dL)', 'D-dimer (mg/L FEU)']

Overall label distribution:


,Prognosis_label,count
0,0,232
1,1,98


Overall Prognosis_label=1 fraction: 0.2970

Tumor variable table aligned to patient list:


,Patient_set,Patient_index,Tumor_AP_diameter_mm,Tumor_longitudinal_diameter_mm,Tumor_transverse_diameter_mm,Tumor_volume_mm3
0,set_1,1,122.987806,144.968236,116.466989,816281.627888
1,set_1,5,79.927032,89.614892,79.265397,137699.289711
2,set_1,7,90.605079,105.821709,98.946985,267149.414439
3,set_1,8,40.059695,48.504358,41.457716,30504.607393
4,set_1,11,89.836784,89.984684,91.460815,362204.846620


In [3]:
# ============================================================
# Helper functions for tumor-size p-value balance
# ============================================================

def continuous_p_value_for_label(df_eval, value_col, label_col=label_col):
    """Compare one continuous tumor variable between label=0 and label=1."""
    g0 = pd.to_numeric(df_eval.loc[df_eval[label_col] == 0, value_col], errors="coerce").dropna().to_numpy(float)
    g1 = pd.to_numeric(df_eval.loc[df_eval[label_col] == 1, value_col], errors="coerce").dropna().to_numpy(float)

    if len(g0) < 3 or len(g1) < 3:
        return np.nan, "too_few"

    try:
        normal0 = stats.shapiro(g0).pvalue > 0.05 if len(g0) <= 5000 else False
        normal1 = stats.shapiro(g1).pvalue > 0.05 if len(g1) <= 5000 else False
    except Exception:
        normal0 = False
        normal1 = False

    if normal0 and normal1:
        p = stats.ttest_ind(g0, g1, equal_var=False, nan_policy="omit").pvalue
        return float(p), "Welch_t"

    p = stats.mannwhitneyu(g0, g1, alternative="two-sided").pvalue
    return float(p), "Mann_Whitney_U"


def tumor_p_table(df_eval, dataset_name):
    rows = []
    for var in tumor_variable_cols:
        p_value, method = continuous_p_value_for_label(df_eval, var)
        rows.append({
            "Dataset": dataset_name,
            "Variable": var,
            "n": int(df_eval.shape[0]),
            "n_label0": int((df_eval[label_col] == 0).sum()),
            "n_label1": int((df_eval[label_col] == 1).sum()),
            "P_value": p_value,
            "P_method": method,
            "Significant_p_lt_0_05": bool(pd.notna(p_value) and p_value < tumor_p_alpha),
        })
    return pd.DataFrame(rows)


def add_tumor_columns(df_base):
    """Return a temporary dataframe with tumor variables added by row order."""
    df_tmp = df_base.copy()
    for var in tumor_variable_cols:
        df_tmp[var] = tumor_values[var].to_numpy()
    return df_tmp


def split_pattern_matches(train_df, test_df):
    train_p = tumor_p_table(train_df, "train")
    test_p = tumor_p_table(test_df, "internal test")
    merged_p = train_p[["Variable", "P_value", "P_method", "Significant_p_lt_0_05"]].merge(
        test_p[["Variable", "P_value", "P_method", "Significant_p_lt_0_05"]],
        on="Variable",
        suffixes=("_train", "_internal_test"),
    )
    merged_p["Pattern_match"] = merged_p["Significant_p_lt_0_05_train"] == merged_p["Significant_p_lt_0_05_internal_test"]
    return bool(merged_p["Pattern_match"].all()), merged_p


def evaluate_fold_seed(df_split_base, fold_random_state):
    """Evaluate one StratifiedKFold seed.

    For each validation fold, compare the tumor-significance pattern in:
    - cv_train: the other 4 folds
    - cv_val: the held-out fold

    A mismatch means a variable is significant in one side but not the other.
    """
    df_eval = df_split_base.copy()
    df_eval[fold_col] = -1

    train_mask = df_eval[split_col] == "train"
    train_indices_ordered = df_eval.index[train_mask].to_numpy()
    y_train = df_eval.loc[train_indices_ordered, label_col].astype(int).values

    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=fold_random_state)
    for fold_id, (_, val_pos) in enumerate(skf.split(train_indices_ordered, y_train)):
        val_indices = train_indices_ordered[val_pos]
        df_eval.loc[val_indices, fold_col] = fold_id

    df_eval.loc[df_eval[split_col] == "internal test", fold_col] = n_splits

    detailed_rows = []
    mismatch_count = 0

    for fold_id in range(n_splits):
        cv_train_df = df_eval[(df_eval[split_col] == "train") & (df_eval[fold_col] != fold_id)].copy()
        cv_val_df = df_eval[(df_eval[split_col] == "train") & (df_eval[fold_col] == fold_id)].copy()

        cv_train_p = tumor_p_table(cv_train_df, f"seed{fold_random_state}_fold{fold_id}_cv_train")
        cv_val_p = tumor_p_table(cv_val_df, f"seed{fold_random_state}_fold{fold_id}_cv_val")

        merged_p = cv_train_p[["Variable", "P_value", "P_method", "Significant_p_lt_0_05"]].merge(
            cv_val_p[["Variable", "P_value", "P_method", "Significant_p_lt_0_05"]],
            on="Variable",
            suffixes=("_cv_train", "_cv_val"),
        )
        merged_p["Pattern_match"] = merged_p["Significant_p_lt_0_05_cv_train"] == merged_p["Significant_p_lt_0_05_cv_val"]
        merged_p["Fold"] = fold_id
        merged_p["fold_random_state"] = fold_random_state
        merged_p["n_cv_train"] = int(cv_train_df.shape[0])
        merged_p["n_cv_val"] = int(cv_val_df.shape[0])

        mismatch_count += int((~merged_p["Pattern_match"]).sum())
        detailed_rows.append(merged_p)

    detailed_df = pd.concat(detailed_rows, ignore_index=True)
    return mismatch_count, detailed_df, df_eval


In [4]:
# ============================================================
# Fixed train/internal-test split with tumor p-value pattern matching
# Mode 1: search  -> search seed automatically
# Mode 2: manual  -> use known seed directly and print p values
# ============================================================

split_mode = "manual"   # choose: "search" or "manual"
manual_split_random_state = 171  # used only when split_mode == "manual"

if split_mode not in ["search", "manual"]:
    raise ValueError(f"split_mode must be 'search' or 'manual'. Got: {split_mode}")

all_indices = np.arange(len(df0))
y_all = df0[label_col].values

selected_split_random_state = None
selected_split_p_table = None
selected_train_idx = None
selected_internal_test_idx = None


# ============================================================
# Helper: evaluate one train/internal-test split seed
# ============================================================

def evaluate_train_internal_split_seed(candidate_seed):
    train_idx, internal_test_idx = train_test_split(
        all_indices,
        train_size=n_train,
        test_size=n_internal_test,
        stratify=y_all,
        random_state=candidate_seed,
        shuffle=True,
    )

    df_candidate = df0.copy()
    df_candidate[split_col] = ""

    df_candidate.loc[train_idx, split_col] = "train"
    df_candidate.loc[internal_test_idx, split_col] = "internal test"

    df_candidate_with_tumor = add_tumor_columns(df_candidate)

    train_df = df_candidate_with_tumor[
        df_candidate_with_tumor[split_col] == "train"
    ].copy()

    test_df = df_candidate_with_tumor[
        df_candidate_with_tumor[split_col] == "internal test"
    ].copy()

    match_ok, split_p_table = split_pattern_matches(train_df, test_df)

    return train_idx, internal_test_idx, match_ok, split_p_table


# ============================================================
# Mode 1: search seed
# ============================================================

if split_mode == "search":
    print("Searching train/internal-test split...")
    print("Requirement: for each tumor variable, train and internal test must have the same p<0.05 status.")

    for attempt_i, candidate_seed in enumerate(
        range(split_random_state, split_random_state + max_split_search_attempts),
        start=1,
    ):
        (
            train_idx,
            internal_test_idx,
            match_ok,
            split_p_table,
        ) = evaluate_train_internal_split_seed(candidate_seed)

        if attempt_i == 1 or attempt_i % 500 == 0 or match_ok:
            mismatch_num = int((~split_p_table["Pattern_match"]).sum())
            print(f"  attempt={attempt_i}, seed={candidate_seed}, mismatch={mismatch_num}")

        if match_ok:
            selected_split_random_state = candidate_seed
            selected_split_p_table = split_p_table
            selected_train_idx = train_idx
            selected_internal_test_idx = internal_test_idx
            break

    if selected_split_random_state is None:
        raise RuntimeError(
            f"No train/internal-test split satisfied tumor p-value pattern matching within {max_split_search_attempts} attempts. "
            "Increase max_split_search_attempts or relax the criterion."
        )


# ============================================================
# Mode 2: manually use known seed
# ============================================================

if split_mode == "manual":
    print("Using manually assigned train/internal-test split seed:")
    print("manual_split_random_state =", manual_split_random_state)

    (
        train_idx,
        internal_test_idx,
        match_ok,
        split_p_table,
    ) = evaluate_train_internal_split_seed(manual_split_random_state)

    mismatch_num = int((~split_p_table["Pattern_match"]).sum())

    print("Manual split tumor p-value pattern mismatch count:", mismatch_num)
    print("Manual split pattern match:", match_ok)

    selected_split_random_state = manual_split_random_state
    selected_split_p_table = split_p_table
    selected_train_idx = train_idx
    selected_internal_test_idx = internal_test_idx


# ============================================================
# Build fixed split dataframe
# ============================================================

split_random_state = selected_split_random_state
train_idx = selected_train_idx
internal_test_idx = selected_internal_test_idx

df_split_base = df0.copy()
df_split_base[split_col] = ""

df_split_base.loc[train_idx, split_col] = "train"
df_split_base.loc[internal_test_idx, split_col] = "internal test"

if (df_split_base[split_col] == "").any():
    raise RuntimeError("Some cases were not assigned to train/internal test.")

print("\nFixed split created with split_random_state =", split_random_state)
print(df_split_base[split_col].value_counts())

split_summary = (
    df_split_base
    .groupby(split_col)[label_col]
    .agg(
        n="count",
        positive_count="sum",
        positive_fraction="mean",
    )
    .reset_index()
)

print("\nTrain/internal-test label summary:")
display(split_summary)

print("\nTrain/internal-test tumor p-value pattern:")
display(selected_split_p_table)

# Optional: check exact index stability for later use.
fixed_train_indices = set(train_idx.tolist())
fixed_internal_test_indices = set(internal_test_idx.tolist())

Using manually assigned train/internal-test split seed:
manual_split_random_state = 171
Manual split tumor p-value pattern mismatch count: 0
Manual split pattern match: True

Fixed split created with split_random_state = 171
train            232
internal test     98
Name: split, dtype: int64

Train/internal-test label summary:


,split,n,positive_count,positive_fraction
0,internal test,98,29,0.295918
1,train,232,69,0.297414



Train/internal-test tumor p-value pattern:


,Variable,P_value_train,P_method_train,Significant_p_lt_0_05_train,P_value_internal_test,P_method_internal_test,Significant_p_lt_0_05_internal_test,Pattern_match
0,Tumor_AP_diameter_mm,0.039842,Mann_Whitney_U,True,0.014530,Mann_Whitney_U,True,True
1,Tumor_longitudinal_diameter_mm,0.074836,Mann_Whitney_U,False,0.123306,Mann_Whitney_U,False,True
2,Tumor_transverse_diameter_mm,0.029216,Mann_Whitney_U,True,0.016174,Mann_Whitney_U,True,True
3,Tumor_volume_mm3,0.043044,Mann_Whitney_U,True,0.017979,Mann_Whitney_U,True,True


In [10]:
# ============================================================
# Search 5-fold split seeds only, then print selected seed values
# ============================================================

print("Searching fold random seeds...")
print("Goal: minimize mismatch between each validation fold and its 4-fold CV-training subset")
print("      for the p<0.05 status of each tumor variable.")

df_split_base_with_tumor = add_tumor_columns(df_split_base)

selected_fold_seed_records = []
selected_fold_detail_tables = []
candidate_records = []

for candidate_seed in range(
    fold_seed_search_start,
    fold_seed_search_start + max_fold_seed_search_attempts,
):
    mismatch_count, detail_df, _ = evaluate_fold_seed(
        df_split_base_with_tumor,
        candidate_seed,
    )

    candidate_records.append({
        "fold_random_state": candidate_seed,
        "mismatch_count": mismatch_count,
    })

    if mismatch_count == 0:
        selected_fold_seed_records.append({
            "fold_random_state": candidate_seed,
            "mismatch_count": mismatch_count,
            "selection_reason": "perfect_zero_mismatch",
        })
        selected_fold_detail_tables.append(detail_df)

        print(f"  selected perfect seed = {candidate_seed}, mismatch = 0")

        if len(selected_fold_seed_records) >= num_fold_random_states_to_keep:
            break

    if (candidate_seed - fold_seed_search_start + 1) % 1000 == 0:
        best_so_far = min(r["mismatch_count"] for r in candidate_records)
        print(
            f"  searched {candidate_seed - fold_seed_search_start + 1} seeds; "
            f"best mismatch so far = {best_so_far}"
        )

# If perfect seeds are not enough, select the lowest-mismatch seeds.
if len(selected_fold_seed_records) < num_fold_random_states_to_keep:
    print("\nNot enough perfect fold seeds found. Selecting lowest-mismatch seeds instead.")

    candidate_df = pd.DataFrame(candidate_records).sort_values(
        ["mismatch_count", "fold_random_state"],
        ascending=[True, True],
    )

    already_selected = {
        r["fold_random_state"]
        for r in selected_fold_seed_records
    }

    for _, row in candidate_df.iterrows():
        candidate_seed = int(row["fold_random_state"])

        if candidate_seed in already_selected:
            continue

        mismatch_count, detail_df, _ = evaluate_fold_seed(
            df_split_base_with_tumor,
            candidate_seed,
        )

        selected_fold_seed_records.append({
            "fold_random_state": candidate_seed,
            "mismatch_count": int(mismatch_count),
            "selection_reason": "lowest_mismatch_available",
        })
        selected_fold_detail_tables.append(detail_df)
        already_selected.add(candidate_seed)

        print(f"  selected seed = {candidate_seed}, mismatch = {mismatch_count}")

        if len(selected_fold_seed_records) >= num_fold_random_states_to_keep:
            break

fold_seed_summary_df = pd.DataFrame(selected_fold_seed_records)
fold_random_state_list = fold_seed_summary_df["fold_random_state"].astype(int).tolist()

print("\n============================================================")
print("Selected fold random states:")
print(fold_random_state_list)

print("\nSelected seed details:")
display(fold_seed_summary_df)

print("\nTumor p-value details for selected seeds:")
if len(selected_fold_detail_tables) > 0:
    selected_fold_detail_df = pd.concat(
        selected_fold_detail_tables,
        ignore_index=True,
    )
    display(selected_fold_detail_df)

Searching fold random seeds...
Goal: minimize mismatch between each validation fold and its 4-fold CV-training subset
      for the p<0.05 status of each tumor variable.
  searched 1000 seeds; best mismatch so far = 3
  searched 2000 seeds; best mismatch so far = 3
  searched 3000 seeds; best mismatch so far = 3
  searched 4000 seeds; best mismatch so far = 3
  searched 5000 seeds; best mismatch so far = 2
  searched 6000 seeds; best mismatch so far = 2
  searched 7000 seeds; best mismatch so far = 2
  searched 8000 seeds; best mismatch so far = 2
  searched 9000 seeds; best mismatch so far = 2
  searched 10000 seeds; best mismatch so far = 2
  searched 11000 seeds; best mismatch so far = 2
  searched 12000 seeds; best mismatch so far = 2
  searched 13000 seeds; best mismatch so far = 2
  searched 14000 seeds; best mismatch so far = 2
  searched 15000 seeds; best mismatch so far = 2
  searched 16000 seeds; best mismatch so far = 2
  searched 17000 seeds; best mismatch so far = 2
  sear

,fold_random_state,mismatch_count,selection_reason
0,4285,2,lowest_mismatch_available
1,6028,2,lowest_mismatch_available
2,13845,2,lowest_mismatch_available
3,14122,2,lowest_mismatch_available
4,14322,2,lowest_mismatch_available



Tumor p-value details for selected seeds:


,Variable,P_value_cv_train,P_method_cv_train,Significant_p_lt_0_05_cv_train,P_value_cv_val,P_method_cv_val,Significant_p_lt_0_05_cv_val,Pattern_match,Fold,fold_random_state,n_cv_train,n_cv_val
0,Tumor_AP_diameter_mm,0.142678,Mann_Whitney_U,False,0.114909,Welch_t,False,True,0,4285,185,47
1,Tumor_longitudinal_diameter_mm,0.067346,Mann_Whitney_U,False,0.641369,Welch_t,False,True,0,4285,185,47
2,Tumor_transverse_diameter_mm,0.073646,Mann_Whitney_U,False,0.145979,Mann_Whitney_U,False,True,0,4285,185,47
3,Tumor_volume_mm3,0.137832,Mann_Whitney_U,False,0.145979,Mann_Whitney_U,False,True,0,4285,185,47
4,Tumor_AP_diameter_mm,0.052685,Mann_Whitney_U,False,0.972165,Mann_Whitney_U,False,True,1,4285,185,47
...,...,...,...,...,...,...,...,...,...,...,...,...
95,Tumor_volume_mm3,0.108364,Mann_Whitney_U,False,0.256813,Mann_Whitney_U,False,True,3,14322,186,46
96,Tumor_AP_diameter_mm,0.086982,Mann_Whitney_U,False,0.299054,Mann_Whitney_U,False,True,4,14322,186,46
97,Tumor_longitudinal_diameter_mm,0.143239,Mann_Whitney_U,False,0.263272,Welch_t,False,True,4,14322,186,46
98,Tumor_transverse_diameter_mm,0.068461,Mann_Whitney_U,False,0.266963,Mann_Whitney_U,False,True,4,14322,186,46


In [7]:
# ============================================================
# Generate 5-fold split files from manually entered random seeds
# ============================================================

manual_fold_random_state_list = [0,10,20,30,40]  # 上面cell搜索出来是[4285, 6028, 13845, 14122, 14322]  

print("Using manually entered fold random states:")
print(manual_fold_random_state_list)

saved_paths = []
saved_fold_detail_tables = []

df_split_base_with_tumor = add_tumor_columns(df_split_base)

for fold_random_state in manual_fold_random_state_list:
    print("\n============================================================")
    print("Generating:", task, "fold_random_state =", fold_random_state)

    mismatch_count, detail_df, df_eval_with_tumor = evaluate_fold_seed(
        df_split_base_with_tumor,
        fold_random_state,
    )

    print("Fold seed mismatch count:", mismatch_count)

    df_out = df_eval_with_tumor.drop(
        columns=tumor_variable_cols,
        errors="ignore",
    ).copy()

    train_mask = df_out[split_col] == "train"
    internal_test_mask = df_out[split_col] == "internal test"

    if (df_out[fold_col] < 0).any():
        raise RuntimeError(f"Unassigned fold exists for random_state={fold_random_state}")

    if not (df_out.loc[internal_test_mask, fold_col] == n_splits).all():
        raise RuntimeError("Internal test fold assignment is not fixed as 5.")

    if not set(df_out.loc[train_mask, fold_col].unique()).issubset(set(range(n_splits))):
        raise RuntimeError("Train fold assignment contains values outside 0-4.")

    current_train_indices = set(df_out.index[df_out[split_col] == "train"].tolist())
    current_internal_test_indices = set(
        df_out.index[df_out[split_col] == "internal test"].tolist()
    )

    if current_train_indices != fixed_train_indices:
        raise RuntimeError("Train split changed unexpectedly.")

    if current_internal_test_indices != fixed_internal_test_indices:
        raise RuntimeError("Internal test split changed unexpectedly.")

    print("\nSplit summary:")
    display(
        df_out
        .groupby(split_col)[label_col]
        .agg(
            n="count",
            positive_count="sum",
            positive_fraction="mean",
        )
        .reset_index()
    )

    print("\nFold summary:")
    fold_summary = (
        df_out
        .groupby([split_col, fold_col])[label_col]
        .agg(
            n="count",
            positive_count="sum",
            positive_fraction="mean",
        )
        .reset_index()
        .sort_values([fold_col, split_col])
    )
    display(fold_summary)

    print("\nTumor p-values for each validation fold and its CV-training subset:")
    display(detail_df)

    detail_save = detail_df.copy()
    detail_save["selected_split_random_state"] = split_random_state
    detail_save["fold_random_state"] = fold_random_state
    saved_fold_detail_tables.append(detail_save)

    out_path = os.path.join(
        out_root,
        f"image_label_info_set12_5fold_{task}_random{fold_random_state}.xlsx",
    )

    df_out.to_excel(out_path, index=False)
    saved_paths.append(out_path)

    print("Saved:", out_path)


# Save audit report for the generated split files.
with pd.ExcelWriter(tumor_balance_report_path) as writer:
    selected_split_p_table.to_excel(
        writer,
        sheet_name="train_internal_test",
        index=False,
    )

    pd.DataFrame({
        "fold_random_state": manual_fold_random_state_list,
    }).to_excel(
        writer,
        sheet_name="manual_fold_seeds",
        index=False,
    )

    pd.concat(
        saved_fold_detail_tables,
        ignore_index=True,
    ).to_excel(
        writer,
        sheet_name="fold_p_values",
        index=False,
    )

print("\nSaved tumor balance report:")
print(tumor_balance_report_path)

Using manually entered fold random states:
[0, 10, 20, 30, 40]

Generating: prognosis fold_random_state = 0
Fold seed mismatch count: 8

Split summary:


,split,n,positive_count,positive_fraction
0,internal test,98,29,0.295918
1,train,232,69,0.297414



Fold summary:


,split,fold,n,positive_count,positive_fraction
1,train,0,47,14,0.297872
2,train,1,47,14,0.297872
3,train,2,46,13,0.282609
4,train,3,46,14,0.304348
5,train,4,46,14,0.304348
0,internal test,5,98,29,0.295918



Tumor p-values for each validation fold and its CV-training subset:


...


Fold summary:


,split,fold,n,positive_count,positive_fraction
1,train,0,47,14,0.297872
2,train,1,47,14,0.297872
3,train,2,46,13,0.282609
4,train,3,46,14,0.304348
5,train,4,46,14,0.304348
0,internal test,5,98,29,0.295918



Tumor p-values for each validation fold and its CV-training subset:


,Variable,P_value_cv_train,P_method_cv_train,Significant_p_lt_0_05_cv_train,P_value_cv_val,P_method_cv_val,Significant_p_lt_0_05_cv_val,Pattern_match,Fold,fold_random_state,n_cv_train,n_cv_val
0,Tumor_AP_diameter_mm,0.036287,Mann_Whitney_U,True,0.911367,Welch_t,False,False,0,40,185,47
1,Tumor_longitudinal_diameter_mm,0.087413,Mann_Whitney_U,False,0.602076,Welch_t,False,True,0,40,185,47
2,Tumor_transverse_diameter_mm,0.023610,Mann_Whitney_U,True,0.867964,Welch_t,False,False,0,40,185,47
3,Tumor_volume_mm3,0.080941,Mann_Whitney_U,False,0.221987,Mann_Whitney_U,False,True,0,40,185,47
4,Tumor_AP_diameter_mm,0.097897,Mann_Whitney_U,False,0.173566,Mann_Whitney_U,False,True,1,40,185,47
5,Tumor_longitudinal_diameter_mm,0.066452,Mann_Whitney_U,False,0.606942,Welch_t,False,True,1,40,185,47
6,Tumor_transverse_diameter_mm,0.088813,Mann_Whitney_U,False,0.145979,Mann_Whitney_U,False,True,1,40,185,47
7,Tumor_volume_mm3,0.102869,Mann_Whitney_U,False,0.188742,Mann_Whitney_U,False,True,1,40,185,47
8,Tumor_AP_diameter_mm,0.041531,Mann_Whitney_U,True,0.479271,Mann_Whitney_U,False,False,2,40,186,46
9,Tumor_longitudinal_diameter_mm,0.224065,Mann_Whitney_U,False,0.088992,Welch_t,False,True,2,40,186,46


Saved: /host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_random40.xlsx

Saved tumor balance report:
/host/e/D/Data/Habitats/Jishuitan/Patient_lists/image_label_info_set12_5fold_prognosis_tumor_balance_report.xlsx


In [6]:
# ============================================================
# Verify saved files
# ============================================================

print("\n============================================================")
print("Verifying saved split files...")

reference_df = pd.read_excel(saved_paths[0])
reference_split = reference_df[split_col].astype(str).tolist()

verify_rows = []

for path in saved_paths:
    df_check = pd.read_excel(path)

    same_split = df_check[split_col].astype(str).tolist() == reference_split
    internal_test_fold_ok = (
        df_check.loc[df_check[split_col] == "internal test", fold_col] == n_splits
    ).all()

    train_n = int((df_check[split_col] == "train").sum())
    internal_test_n = int((df_check[split_col] == "internal test").sum())

    train_pos_frac = float(
        df_check.loc[df_check[split_col] == "train", label_col].astype(int).mean()
    )
    internal_test_pos_frac = float(
        df_check.loc[df_check[split_col] == "internal test", label_col].astype(int).mean()
    )

    verify_rows.append(
        {
            "file": os.path.basename(path),
            "same_split_as_first_file": same_split,
            "internal_test_fold_is_5": internal_test_fold_ok,
            "train_n": train_n,
            "internal_test_n": internal_test_n,
            "train_positive_fraction": train_pos_frac,
            "internal_test_positive_fraction": internal_test_pos_frac,
        }
    )

verify_df = pd.DataFrame(verify_rows)
display(verify_df)

if not verify_df["same_split_as_first_file"].all():
    raise RuntimeError("Not all files have the same fixed train/internal-test split.")

if not verify_df["internal_test_fold_is_5"].all():
    raise RuntimeError("Not all files have internal test fold fixed as 5.")

print("All saved files verified.")


Verifying saved split files...


,file,same_split_as_first_file,internal_test_fold_is_5,train_n,internal_test_n,train_positive_fraction,internal_test_positive_fraction
0,image_label_info_set12_5fold_prognosis_random0...,True,True,232,98,0.297414,0.295918
1,image_label_info_set12_5fold_prognosis_random1...,True,True,232,98,0.297414,0.295918
2,image_label_info_set12_5fold_prognosis_random2...,True,True,232,98,0.297414,0.295918
3,image_label_info_set12_5fold_prognosis_random3...,True,True,232,98,0.297414,0.295918
4,image_label_info_set12_5fold_prognosis_random4...,True,True,232,98,0.297414,0.295918
5,image_label_info_set12_5fold_prognosis_random5...,True,True,232,98,0.297414,0.295918


All saved files verified.


In [ ]:
import os
import pandas as pd


# ============================================================
# Settings
# ============================================================

patient_list_dir = "/host/e/D/Data/Habitats/Jishuitan/Patient_lists"

split_file = os.path.join(
    patient_list_dir,
    "image_label_info_set12_5fold_prognosis_random0.xlsx",
)

bbox_file = os.path.join(
    patient_list_dir,
    "image_label_info_set12_resampled_bbox_1x1x3.xlsx",
)

save_file = os.path.join(
    patient_list_dir,
    "image_label_info_set12_5fold_prognosis_random0_v2.xlsx",
)


# ============================================================
# Bbox inclusion range
# You can edit these values.
# Keep case only if:
#   x_min <= bbox_x_shape <= x_max
#   y_min <= bbox_y_shape <= y_max
#   z_min <= bbox_z_shape <= z_max
# ============================================================

bbox_x_min = 50
bbox_x_max = 96

bbox_y_min = 50
bbox_y_max = 96

bbox_z_min = 30
bbox_z_max = 60


# ============================================================
# Load tables
# ============================================================

split_df = pd.read_excel(split_file)
bbox_df = pd.read_excel(bbox_file)

print("Original split table shape:", split_df.shape)
print("Bbox table shape:", bbox_df.shape)

required_split_cols = ["Patient_set", "Patient_index"]
required_bbox_cols = [
    "Patient_set",
    "Patient_index",
    "bbox_x_shape",
    "bbox_y_shape",
    "bbox_z_shape",
]

for col in required_split_cols:
    if col not in split_df.columns:
        raise KeyError(f"Missing column in split table: {col}")

for col in required_bbox_cols:
    if col not in bbox_df.columns:
        raise KeyError(f"Missing column in bbox table: {col}")


# ============================================================
# Make patient ids consistent
# ============================================================

split_df["Patient_set"] = split_df["Patient_set"].astype(str)
split_df["Patient_index"] = split_df["Patient_index"].astype(str)

bbox_df["Patient_set"] = bbox_df["Patient_set"].astype(str)
bbox_df["Patient_index"] = bbox_df["Patient_index"].astype(str)


# ============================================================
# Merge bbox info into split table
# ============================================================

bbox_keep_cols = [
    "Patient_set",
    "Patient_index",
    "bbox_x_shape",
    "bbox_y_shape",
    "bbox_z_shape",
]

merged_df = split_df.merge(
    bbox_df[bbox_keep_cols],
    on=["Patient_set", "Patient_index"],
    how="left",
    validate="one_to_one",
)

missing_bbox_num = merged_df["bbox_x_shape"].isna().sum()
print("Cases without bbox info:", missing_bbox_num)

if missing_bbox_num > 0:
    missing_cases = merged_df.loc[
        merged_df["bbox_x_shape"].isna(),
        ["Patient_set", "Patient_index"],
    ]
    print("Missing bbox cases:")
    print(missing_cases)


# ============================================================
# Filter by bbox shape range
# ============================================================

keep_mask = (
    merged_df["bbox_x_shape"].between(bbox_x_min, bbox_x_max, inclusive="both")
    & merged_df["bbox_y_shape"].between(bbox_y_min, bbox_y_max, inclusive="both")
    & merged_df["bbox_z_shape"].between(bbox_z_min, bbox_z_max, inclusive="both")
)

filtered_df = merged_df[keep_mask].copy()
removed_df = merged_df[~keep_mask].copy()

print("\n============================================================")
print("Filtering criteria")
print("bbox_x:", bbox_x_min, "to", bbox_x_max)
print("bbox_y:", bbox_y_min, "to", bbox_y_max)
print("bbox_z:", bbox_z_min, "to", bbox_z_max)

print("\nOriginal case number:", merged_df.shape[0])
print("Kept case number:", filtered_df.shape[0])
print("Removed case number:", removed_df.shape[0])


# ============================================================
# Print label and fold distribution before/after
# ============================================================

label_cols = [
    col for col in ["Pathologic_label", "Prognosis_label"]
    if col in merged_df.columns
]

for label_col in label_cols:
    print("\n============================================================")
    print("Label distribution:", label_col)

    print("\nBefore filtering:")
    print(merged_df[label_col].value_counts(dropna=False).sort_index())
    print("Positive fraction:", merged_df[label_col].mean())

    print("\nAfter filtering:")
    print(filtered_df[label_col].value_counts(dropna=False).sort_index())
    print("Positive fraction:", filtered_df[label_col].mean())

if "fold" in merged_df.columns:
    print("\n============================================================")
    print("Fold distribution before filtering:")
    print(merged_df["fold"].value_counts(dropna=False).sort_index())

    print("\nFold distribution after filtering:")
    print(filtered_df["fold"].value_counts(dropna=False).sort_index())

if "split" in merged_df.columns:
    print("\n============================================================")
    print("Split distribution before filtering:")
    print(merged_df["split"].value_counts(dropna=False))

    print("\nSplit distribution after filtering:")
    print(filtered_df["split"].value_counts(dropna=False))


# ============================================================
# Optional: show removed cases
# ============================================================

print("\n============================================================")
print("Removed cases:")
show_cols = [
    "Patient_set",
    "Patient_index",
    "bbox_x_shape",
    "bbox_y_shape",
    "bbox_z_shape",
]

for label_col in label_cols:
    show_cols.append(label_col)

if "fold" in removed_df.columns:
    show_cols.append("fold")

if "split" in removed_df.columns:
    show_cols.append("split")

print(removed_df[show_cols].to_string(index=False))


# ============================================================
# Save
# Keep bbox columns in output so you can verify filtering later.
# If you do not want bbox columns in the saved file, drop them before saving.
# ============================================================

filtered_df.to_excel(save_file, index=False)

print("\n============================================================")
print("Saved filtered patient list:")
print(save_file)